In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install ultralytics

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO

In [ ]:
train_csv = "/kaggle/input/datasets/alaagastien/cliniscan2/train.csv"
meta_csv = "/kaggle/input/datasets/alaagastien/cliniscan2/archive/train_meta.csv"

df = pd.read_csv(train_csv)
meta = pd.read_csv(meta_csv)

print("Train rows:", len(df))
print("Meta rows:", len(meta))

In [ ]:
df = df[df["class_id"] != 14]

print("After removing No finding:", len(df))

In [ ]:
df = df.dropna(subset=["x_min","y_min","x_max","y_max"])

print("Valid annotations:", len(df))

In [ ]:
size_dict = {}

for _, row in meta.iterrows():
    size_dict[row["image_id"]] = (row["dim0"], row["dim1"])

In [ ]:
base = "/kaggle/working/dataset"

img_dir = f"{base}/images/train"
lbl_dir = f"{base}/labels/train"

os.makedirs(img_dir, exist_ok=True)
os.makedirs(lbl_dir, exist_ok=True)

In [ ]:
for img_id, group in tqdm(df.groupby("image_id")):

    h, w = size_dict[img_id]

    label_path = os.path.join(lbl_dir, img_id + ".txt")

    with open(label_path, "w") as f:

        for _, row in group.iterrows():

            x_min = row["x_min"]
            y_min = row["y_min"]
            x_max = row["x_max"]
            y_max = row["y_max"]

            xc = ((x_min + x_max) / 2) / w
            yc = ((y_min + y_max) / 2) / h

            bw = (x_max - x_min) / w
            bh = (y_max - y_min) / h

            class_id = int(row["class_id"])

            f.write(f"{class_id} {xc} {yc} {bw} {bh}\n")

In [ ]:
src_img_dir = "/kaggle/input/datasets/alaagastien/cliniscan2/archive/train"

for img in tqdm(os.listdir(src_img_dir)):
    
    src = os.path.join(src_img_dir, img)
    dst = os.path.join(img_dir, img)

    if not os.path.exists(dst):
        os.system(f"cp {src} {dst}")

In [ ]:
images = set([os.path.splitext(f)[0] for f in os.listdir(img_dir)])
labels = set([os.path.splitext(f)[0] for f in os.listdir(lbl_dir)])

missing = images - labels

for img in missing:
    open(os.path.join(lbl_dir, img + ".txt"), "w").close()

print("Empty labels created:", len(missing))

In [ ]:
print("Images:", len(os.listdir(img_dir)))
print("Labels:", len(os.listdir(lbl_dir)))

In [ ]:
data_yaml = """
train: /kaggle/working/dataset/images/train
val: /kaggle/working/dataset/images/train

nc: 14

names:
  0: Aortic enlargement
  1: Atelectasis
  2: Calcification
  3: Cardiomegaly
  4: Consolidation
  5: ILD
  6: Infiltration
  7: Lung Opacity
  8: Nodule/Mass
  9: Other lesion
  10: Pleural effusion
  11: Pleural thickening
  12: Pneumothorax
  13: Pulmonary fibrosis
"""

with open("/kaggle/working/data.yaml","w") as f:
    f.write(data_yaml)

In [ ]:
model = YOLO("yolo11s.pt")

model.train(
    data="/kaggle/working/data.yaml",
    epochs=25,
    imgsz=1024,
    batch=8
)

In [ ]:
model.predict(
    source="/kaggle/input/datasets/alaagastien/cliniscan2/archive/test",
    save=True,
    conf=0.25
)

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

# Load your trained weights from Milestone 2
# Note: Double check if your path is 'best.pt' or 'last.pt'
model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

In [ ]:
from ultralytics import YOLO

# 1. Load the best weights you've trained
model = YOLO('/kaggle/input/notebook7f8f594ad4/yolo11s.pt')

# 2. Run the prediction on your test images
results = model.predict(
    source="/kaggle/input/cliniscan2/archive/test", 
    save=True,      # This saves the images with bounding boxes
    conf=0.25      # Only show detections the model is >25% sure about
)

In [ ]:
import os
# This searches your entire working directory for any .pt files
for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        if file.endswith(".pt"):
            print(os.path.join(root, file))

In [ ]:
import os

# This looks for any file ending in '.pt' in your working directory
for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        if file.endswith('.pt'):
            print(f"Found your model at: {os.path.join(root, file)}")

In [ ]:
import os
print("--- Checking for Weights ---")
for root, dirs, files in os.walk('/kaggle'):
    for file in files:
        if file.endswith('.pt'):
            print(f"FOUND IT: {os.path.join(root, file)}")

In [ ]:
# 1. Install the library
!pip install ultralytics -q

# 2. Load the model from your specific Output path
from ultralytics import YOLO
# Based on your image 7e312e60, the path to your best weights is:
model = YOLO('/kaggle/working/yolo11s.pt')

In [ ]:
results = model.predict(
    source="/kaggle/input/cliniscan2/archive/test", 
    save=True, 
    conf=0.25
)

In [ ]:
import os
from ultralytics import YOLO

# 1. Check if the test folder exists before running
test_path = "/kaggle/input/cliniscan2/archive/test"

if os.path.exists(test_path):
    print("✅ Test folder found! Starting prediction...")
    
    # 2. Load the model from your current working directory
    model = YOLO('/kaggle/working/yolo11s.pt')

    # 3. Run the prediction with the CORRECT absolute path
    results = model.predict(
        source=test_path, 
        save=True, 
        conf=0.25
    )
    print("🚀 Done! Check the 'predict' folder in your Output.")
else:
    print("❌ Error: Path not found. Run the code below to see the real path.")
    !ls /kaggle/input/cliniscan2/archive/

In [8]:
!pip install torch torchvision timm

In [9]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [10]:
train_csv = "/kaggle/input/datasets/alaagastien/cliniscan2/train.csv"
img_dir = "/kaggle/input/datasets/alaagastien/cliniscan2/archive/train"
df = pd.read_csv(train_csv)

In [11]:
df = df[df["class_id"] != 14]

In [12]:
# Create Image  Single Label Mapping
df = df.sort_values("image_id")

df_single = df.groupby("image_id").first().reset_index()

print("Total images:", len(df_single))

Total images: 4394


In [13]:
class XrayDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = os.path.join(self.img_dir, row["image_id"] + ".png")
        image = Image.open(img_path).convert("RGB")
        
        label = int(row["class_id"])
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [18]:
#Transforms
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [20]:
dataset = XrayDataset(df_single, img_dir, transform)

train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [21]:
model = models.resnet50(pretrained=True)

model.fc = nn.Linear(model.fc.in_features, 14)

model = model.cuda()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 216MB/s]


In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [23]:
epochs = 5

for epoch in range(epochs):
    
    model.train()
    total_loss = 0
    
    for images, labels in train_loader:
        
        images = images.cuda()
        labels = labels.cuda()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 559.1797
Epoch 2, Loss: 498.4019
Epoch 3, Loss: 427.6218
Epoch 4, Loss: 309.2437
Epoch 5, Loss: 157.7684


In [24]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import torch.nn.functional as F

In [25]:
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    
    for images, labels in train_loader:
        
        images = images.cuda()
        labels = labels.cuda()
        
        outputs = model(images)
        
        probs = F.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

In [29]:
accuracy = accuracy_score(all_labels, all_preds)
print("Accuracy:", accuracy)

Accuracy: 0.8764223941738735


In [30]:
f1 = f1_score(all_labels, all_preds, average='macro')
print("F1 Score:", f1)


F1 Score: 0.7603925116139695


In [31]:
auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
print("AUC:", auc)

AUC: 0.9947280537921196


In [32]:
torch.save(model.state_dict(), "/kaggle/working/cliniscan_classification_model.pth")

print("Model saved successfully")

Model saved successfully


In [33]:
import os

print(os.listdir("/kaggle/working"))

['cliniscan_classification_model.pth', '.virtual_documents']


In [34]:
from IPython.display import FileLink

FileLink("/kaggle/working/cliniscan_classification_model.pth")

/kaggle/working/cliniscan_classification_model.pth

In [35]:
class_names = {
 0: "Aortic enlargement",
 1: "Atelectasis",
 2: "Calcification",
 3: "Cardiomegaly",
 4: "Consolidation",
 5: "ILD",
 6: "Infiltration",
 7: "Lung Opacity",
 8: "Nodule/Mass",
 9: "Other lesion",
 10: "Pleural effusion",
 11: "Pleural thickening",
 12: "Pneumothorax",
 13: "Pulmonary fibrosis"
}

import json

with open("/kaggle/working/class_names.json", "w") as f:
    json.dump(class_names, f)

In [36]:
FileLink("/kaggle/working/class_names.json")

/kaggle/working/class_names.json